# Episode 1 — Environment Setup & First Look at the Data

**Course:** Production RAG — YouTube Series  
**GitHub branch:** `episode/01`

This notebook accompanies Episode 1. It walks through:
1. Installing missing dependencies
2. Verifying your environment is correctly configured
3. Your first OpenAI API call
4. Loading our real DHS PDFs and inspecting raw text
5. Seeing the hallucination problem that RAG solves

---
> **Before you start:** Copy `.env.example` to `.env` and fill in your `OPENAI_API_KEY`.

**Our actual dataset — files in `data/raw/`:**

| File | Country | Report |
|------|---------|--------|
| `PR149.pdf` | Ghana | DHS 2022 |
| `PR157.pdf` | Nigeria | DHS 2021 |
| `Final-Mini-DHS-report-FR363.pdf` | Ethiopia | DHS 2019 |
| `FR380.pdf` | Kenya | DHS 2022 Vol I |
| `FR380bis.pdf` | Kenya | DHS 2022 Vol II |
| `FR380erratum.pdf` | Kenya | DHS 2022 Addendum |

## 0. Install dependencies

Run this cell once. It installs everything into the active Codespaces environment.

In [ ]:
import subprocess, sys, os

packages = [
    "python-dotenv",
    "langchain",
    "langchain-openai",
    "langchain-community",
    "pymupdf",
    "pdfplumber",
    "openai",
    "pydantic-settings",
    "rich",
    "tenacity",
    "rank-bm25",
]

print("Installing packages...")
result = subprocess.run(
    [sys.executable, "-m", "pip", "install", "--quiet"] + packages,
    capture_output=True, text=True, cwd="/"
)
if result.returncode == 0:
    print("✅ All packages installed")
else:
    print("❌ Error:", result.stderr[-800:])

Installing packages...
✅ All packages installed


## 1. Environment check

In [ ]:
import json
from pathlib import Path
import os

# ── Ultra-robust repo_root for Codespaces / devcontainers ─────
def get_repo_root():
    """Find the real project root in Codespaces (very common issue)"""
    
    # Priority 1: Standard Codespaces workspace location
    for env_var in ['WORKSPACE_FOLDER', 'VSCODE_CWD', 'CODESPACE_VSCODE_FOLDER']:
        if os.environ.get(env_var):
            p = Path(os.environ[env_var])
            if (p / 'data').exists() and (p / 'src').exists():
                return p.resolve()

    # Priority 2: Look for /workspaces/ folder (most common in Codespaces)
    cwd = Path.cwd()
    for parent in [cwd] + list(cwd.parents):
        if parent.name == 'workspaces' or str(parent).startswith('/workspaces'):
            for child in parent.iterdir():
                if child.is_dir() and (child / 'data' / 'raw').exists():
                    return child
            if (parent / 'womens-health-rag').exists():
                return parent / 'womens-health-rag'

    # Priority 3: Search upward for data/ + notebooks/ markers
    try:
        current = Path.cwd().resolve()
    except:
        current = Path('/workspaces')

    for parent in [current] + list(current.parents):
        if (parent / 'data' / 'raw').exists() and (parent / 'notebooks').exists():
            return parent

    # Priority 4: Fallback using PWD
    pwd = os.environ.get('PWD')
    if pwd:
        p = Path(pwd)
        if (p / 'data').exists():
            return p

    # Last resort
    return Path('/workspaces/womens-health-rag')


repo_root = get_repo_root()
data_dir  = repo_root / 'data' / 'raw'
meta_file = repo_root / 'data' / 'metadata.json'

pdfs = sorted(data_dir.glob('*.pdf'))

# Load metadata map
metadata_map = {}
if meta_file.exists():
    with open(meta_file) as f:
        metadata_map = json.load(f)
    print(f'✅ metadata.json loaded — {len(metadata_map)} entries')

print(f'\n✅ Repo root detected: {repo_root}')
print(f'Data dir: {data_dir}  —  exists: {data_dir.exists()}')
print(f'metadata.json exists: {meta_file.exists()}\n')

print(f'Found {len(pdfs)} PDF(s) in {data_dir}:\n')
print(f'{"File":<48} {"Size":>8}  {"Country":<12} {"Year"}')
print('-' * 85)

for pdf in pdfs:
    size_mb = pdf.stat().st_size / 1024 / 1024
    meta    = metadata_map.get(pdf.stem, {})
    country = meta.get('country', '⚠ no metadata')
    year    = meta.get('year', '')
    print(f'{pdf.name:<48} {size_mb:>7.1f}MB  {country:<12} {year}')

Python 3.12.1 (main, Mar 11 2026, 12:17:56) [GCC 13.3.0]
✅ Python version OK

Repo root: /vscode/bin/linux-x64/8b640eef5a6c6089c029249d48efa5c99adf7d51
src/ path: /vscode/bin/linux-x64/8b640eef5a6c6089c029249d48efa5c99adf7d51/src — exists: False


In [ ]:
# ── Load .env ─────────────────────────────────────────────────
from dotenv import load_dotenv
import os

env_path = repo_root / '.env'
loaded   = load_dotenv(dotenv_path=env_path, override=True)
print(f'Loaded .env from {env_path}: {loaded}')

api_key = os.getenv('OPENAI_API_KEY', '')
if not api_key or not api_key.startswith('sk-'):
    print('❌ OPENAI_API_KEY not set or invalid.')
    print(f'   Expected at: {env_path}')
    print('   Run: cp .env.example .env  — then paste your key.')
else:
    print(f'✅ OPENAI_API_KEY found: sk-...{api_key[-4:]}')

Loaded .env from /vscode/bin/linux-x64/8b640eef5a6c6089c029249d48efa5c99adf7d51/.env: False
✅ OPENAI_API_KEY found: sk-...Z8cA


## 2. Check our PDF dataset

In [27]:
import json
from pathlib import Path
import os

# ── Ultra-robust repo_root detection ─────────────────────────
def get_repo_root():
    """Survive even when os.getcwd() raises FileNotFoundError"""
    candidates = []

    # 1. Try PWD environment variable (most reliable in containers/notebooks)
    pwd = os.environ.get('PWD')
    if pwd:
        candidates.append(Path(pwd))

    # 2. Try __file__ if we're in a script (fallback)
    try:
        candidates.append(Path(__file__).resolve().parent.parent)
    except NameError:
        pass  # We're in a notebook

    # 3. Try common Codespaces / workspace locations
    for env_var in ['VSCODE_CWD', 'WORKSPACE_FOLDER', 'CODESPACE_VSCODE_FOLDER']:
        if os.environ.get(env_var):
            candidates.append(Path(os.environ[env_var]))

    # 4. Last resort: root
    candidates.append(Path('/'))

    for p in candidates:
        try:
            resolved = p.resolve(strict=False)
            # Check for typical project markers
            if (resolved / 'src').exists() and (resolved / 'data').exists():
                return resolved
            if (resolved / '.git').exists():
                return resolved
            if resolved.name.startswith('workspaces') or 'github' in str(resolved).lower():
                if (resolved / 'src').exists():
                    return resolved
        except:
            continue

    # Final fallback - use PWD or root
    return Path(pwd) if pwd else Path('/')


repo_root = get_repo_root()
data_dir  = repo_root / 'data' / 'raw'
meta_file = repo_root / 'data' / 'metadata.json'

pdfs = sorted(data_dir.glob('*.pdf'))

# Load metadata map
metadata_map = {}
if meta_file.exists():
    with open(meta_file) as f:
        metadata_map = json.load(f)
    print(f'✅ metadata.json loaded — {len(metadata_map)} entries')

print(f'\nRepo root detected: {repo_root}')
print(f'Data dir: {data_dir}  —  exists: {data_dir.exists()}')
print(f'metadata.json exists: {meta_file.exists()}\n')

print(f'Found {len(pdfs)} PDF(s) in {data_dir}:\n')
print(f'{"File":<48} {"Size":>8}  {"Country":<12} {"Year"}')
print('-' * 85)

for pdf in pdfs:
    size_mb = pdf.stat().st_size / 1024 / 1024
    meta    = metadata_map.get(pdf.stem, {})
    country = meta.get('country', '⚠ no metadata')
    year    = meta.get('year', '')
    print(f'{pdf.name:<48} {size_mb:>7.1f}MB  {country:<12} {year}')


Repo root detected: /vscode/bin/linux-x64/8b640eef5a6c6089c029249d48efa5c99adf7d51
Data dir: /vscode/bin/linux-x64/8b640eef5a6c6089c029249d48efa5c99adf7d51/data/raw  —  exists: False
metadata.json exists: False

Found 0 PDF(s) in /vscode/bin/linux-x64/8b640eef5a6c6089c029249d48efa5c99adf7d51/data/raw:

File                                                 Size  Country      Year
-------------------------------------------------------------------------------------


## 3. Load our PDFs with the ingestion pipeline

In [29]:
import json
from pathlib import Path
import os

# ── Hardened repo_root for this exact Codespaces environment ─────
def get_repo_root():
    """Strong fix for your current Codespaces setup"""
    
    # 1. Direct match from your workspace (most reliable)
    candidates = [
        Path('/workspaces/womens-health-rag'),           # ← Your actual project
        Path(os.environ.get('WORKSPACE_FOLDER', '')),
        Path(os.environ.get('PWD', '')),
        Path('/workspaces').glob('womens-health-rag*'),  # in case name varies
    ]

    for item in candidates:
        if isinstance(item, Path):
            p = item
        else:
            # Handle glob result
            try:
                p = next(iter(item), None)
            except:
                continue
                
        if p and p.exists() and (p / 'data' / 'raw').exists():
            return p.resolve()

    # Fallback: search from known Codespaces root
    try:
        for p in Path('/workspaces').rglob('womens-health-rag'):
            if p.is_dir() and (p / 'data' / 'raw').exists():
                return p
    except:
        pass

    # Ultimate fallback
    return Path('/workspaces/womens-health-rag')


repo_root = get_repo_root()
data_dir  = repo_root / 'data' / 'raw'
meta_file = repo_root / 'data' / 'metadata.json'

print(f'✅ Repo root detected: {repo_root}')
print(f'Data dir exists: {data_dir.exists()} → {data_dir}')

# === Continue with your code ===
nigeria_pdf = data_dir / 'PR157.pdf'

print(f'\nLooking for: {nigeria_pdf}')
print(f'File exists: {nigeria_pdf.exists()}')

# Load Nigeria DHS (PR157.pdf) — primary demo document
from rag.ingestion.loader import load_pdf

pages = load_pdf(
    nigeria_pdf,
    country='Nigeria',
    year='2021',
    report_type='dhs',
    report_title='Nigeria Demographic and Health Survey 2021',
)

print(f'✅ Loaded {len(pages)} non-empty pages from {nigeria_pdf.name}')
print(f'\nPage 1 metadata:')
print(f'  country:  {pages[0].country}')
print(f'  year:     {pages[0].year}')
print(f'  report:   {pages[0].report_title}')
print(f'  pages:    {pages[0].page_number} of {pages[0].total_pages}')
print(f'\n=== Page 1 raw text (first 1500 chars) ===')
print(pages[0].text[:1500])

✅ Repo root detected: /workspaces/womens-health-rag
Data dir exists: True → /workspaces/womens-health-rag/data/raw

Looking for: /workspaces/womens-health-rag/data/raw/PR157.pdf
File exists: True
✅ Loaded 94 non-empty pages from PR157.pdf

Page 1 metadata:
  country:  Nigeria
  year:     2021
  report:   Nigeria Demographic and Health Survey 2021
  pages:    1 of 100

=== Page 1 raw text (first 1500 chars) ===
 
Nigeria 
Demographic and 
Health Survey 
2023–24 
Key Indicators 
 



In [30]:
# Load ALL reports — note Kenya has 3 files (vol I + vol II + erratum)
from rag.ingestion.loader import load_directory
from collections import Counter

all_pages = load_directory(data_dir, metadata_map=metadata_map)

print(f'Total pages across all reports: {len(all_pages)}')
print()

by_country = Counter(p.country for p in all_pages)
print('Pages per country:')
for country, count in sorted(by_country.items(), key=lambda x: -x[1]):
    bar = '█' * (count // 10)
    print(f'  {country:<15} {count:>4} pages  {bar}')

print()
by_report = Counter(p.report_title for p in all_pages)
print('Pages per report:')
for report, count in sorted(by_report.items(), key=lambda x: -x[1]):
    print(f'  {count:>4}  {report}')

Total pages across all reports: 1466

Pages per country:
                  1466 pages  ██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████

Pages per report:
   684  FR380
   402  FR380bis
   187  Final-Mini-DHS-report-FR363
    94  PR157
    58  PR149
    41  FR380erratum


## 4. Inspect raw text — the artefacts we need to fix

This is the core motivation for Episode 2's cleaner module.

In [33]:
# Pick a page from the middle of the Nigeria report — safely
nigeria_pages = [p for p in all_pages if p.country == 'Kenya']

print(f'Found {len(nigeria_pages)} Nigeria pages in all_pages\n')

if not nigeria_pages:
    print("❌ No Nigeria pages found! Let's debug:")
    print(f"Total pages loaded: {len(all_pages)}")
    if all_pages:
        countries = {p.country for p in all_pages}
        print(f"Available countries: {countries}")
        print("\nFirst 3 pages metadata:")
        for p in all_pages[:3]:
            print(f"  • {p.report_title} | country={p.country} | page={p.page_number}")
    else:
        print("all_pages is empty!")
else:
    # Safe middle page selection
    idx = len(nigeria_pages) // 3
    mid_page = nigeria_pages[idx]

    print(f'Report:  {mid_page.report_title}')
    print(f'Page:    {mid_page.page_number} of {mid_page.total_pages}')
    print(f'Chars:   {len(mid_page.text)}')
    print()
    print('=== RAW TEXT (before any cleaning) ===')
    print(mid_page.text[:2000])

Found 0 Nigeria pages in all_pages

❌ No Nigeria pages found! Let's debug:
Total pages loaded: 1466
Available countries: {''}

First 3 pages metadata:
  • FR380 | country= | page=1
  • FR380 | country= | page=3
  • FR380 | country= | page=4


In [ ]:
# Now clean it and compare
from rag.ingestion.cleaner import clean_page

cleaned = clean_page(mid_page)

print('=== CLEANED TEXT ===')
print(cleaned.text[:2000])
print()
print(f'Raw chars:     {len(mid_page.text):,}')
print(f'Cleaned chars: {len(cleaned.text):,}')
print(f'Noise removed: {(1 - len(cleaned.text)/len(mid_page.text))*100:.1f}%')

## 5. The hallucination problem — why RAG exists

In [ ]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model='gpt-4o-mini', temperature=0)

# Quick smoke test
r = llm.invoke('Say hello in one sentence.')
print(r.content)
print('✅ OpenAI API working')

In [ ]:
# Ask questions about our specific reports — watch the hallucinations
questions = [
    'According to the Nigeria DHS 2021 report (PR157), what is the maternal mortality ratio per 100,000 live births?',
    'What does the Ghana DHS 2022 report (PR149) say about contraceptive prevalence among married women?',
    'According to Kenya DHS 2022 report FR380, what is the under-5 mortality rate?',
]

for i, q in enumerate(questions, 1):
    print(f'━━━ Question {i} ━━━')
    print(f'Q: {q[:90]}...')
    r = llm.invoke(q)
    print(f'A: {r.content[:300]}')
    print()

print('='*60)
print('🚨 The model gave confident answers — but cited nothing.')
print('   It cannot know the exact figures from these specific reports.')
print('   RAG grounds the answer in the actual PDF text.')
print('   Episode 5 shows these same questions answered correctly.')

## 6. Preview: chunking our documents

In [ ]:
# Preview chunking — full walkthrough in Episode 2
from rag.ingestion.cleaner import clean_pages
from rag.ingestion.chunker import ChunkStrategy, chunk_pages, chunk_stats

# Use Nigeria only for the preview
nigeria_cleaned = clean_pages(nigeria_pages)

fixed_chunks     = chunk_pages(nigeria_cleaned, strategy=ChunkStrategy.FIXED,     chunk_size=800, chunk_overlap=150)
recursive_chunks = chunk_pages(nigeria_cleaned, strategy=ChunkStrategy.RECURSIVE, chunk_size=800, chunk_overlap=150)

print('Nigeria DHS — chunking strategy comparison:')
print()
print(f'{"Strategy":<12} {"Chunks":>8} {"Avg chars":>10} {"Min":>6} {"Max":>6}')
print('-' * 46)

for label, chunks in [("FIXED", fixed_chunks), ("RECURSIVE", recursive_chunks)]:
    s = chunk_stats(chunks)
    print(f'{label:<12} {s["total_chunks"]:>8} {s["avg_chars"]:>10} {s["min_chars"]:>6} {s["max_chars"]:>6}')

print()
print('Episode 2 shows WHY recursive outperforms fixed for dense health report prose.')

In [ ]:
# Inspect a real chunk with its metadata
# Choose one from the middle of the document — more interesting than the first
sample = recursive_chunks[min(15, len(recursive_chunks)-1)]

print('=== Sample chunk (recursive) ===')
print(sample.page_content)
print()
print('=== Metadata attached to this chunk ===')
for k, v in sample.metadata.items():
    print(f'  {k:<22} {v}')
print()
print('This metadata is what enables filtered retrieval in Episode 7:')
print('  "Show me data from Nigeria between 2018 and 2022"')
print('  → WHERE country = \'Nigeria\' AND year >= \'2018\' ...')

## ✅ Episode 1 complete

| Step | Status |
|------|--------|
| Dependencies installed | ✅ |
| Python 3.12 confirmed | ✅ |
| OpenAI API connected | ✅ |
| All 6 DHS PDFs loaded | ✅ |
| Metadata map working | ✅ |
| Raw text artefacts seen | ✅ |
| Cleaning pipeline previewed | ✅ |
| Chunking strategies compared | ✅ |
| Hallucination problem demonstrated | ✅ |

## What's next — Episode 2

**Document ingestion & chunking in depth:**
- Fix every artefact type seen above — line by line through `cleaner.py`
- Compare all three chunking strategies with real metrics
- Handle Kenya's multi-volume structure (FR380 + FR380bis + FR380erratum as one logical corpus)
- Build the full clean → chunk → metadata → pgvector pipeline end to end

**GitHub branch:** `episode/02`

---
*Drop questions in the YouTube comments or open a GitHub Discussion.*